# **Project Cycle 3: Two-Sample Inference Analysis**
# **情緒與青少年吸菸行為之雙樣本比例推論分析**
---
* **Group Number (組別):** 10
* **Members (組員):** 
  * 112370234 張智源 (Geraldand) - Project Management & Assumptions
  * 113370201 簡愷毅 (kaibao8171) - Statistical Computing & Data
  * 113370218 彭鈺芳 (Vickyfang77) - Data Visualization & Plotting

## Part 1: Research Question & Variables Definition
## 第一部分：研究問題與變數定義
---
### 1. Research Question (研究問題)
Is the proportion of current cigarette use different between students who felt sad or hopeless and those who did not? (Pre-approved Question 8)

在美國青少年中，過去一年曾感到悲傷或絕望的學生，其目前吸菸的比例是否與未感到悲傷或絕望的學生有顯著差異？

### 2. Variable Definitions (變數定義)
* **Group Variable (群組變數/自變數):** `SadOrHopeless_Recoded` (Binary Categorical)
  * `1` = Felt sad or hopeless (Exposed group / Yes)
  * `0` = Did not feel sad or hopeless (Comparison group / No)
* **Response Variable (反應變數/因變數):** `CurrentCigaretteUse_Recoded` (Binary Categorical)
  * `1` = Current smoker (Smoked on 1 or more days in the past 30 days)
  * `0` = Non-smoker

### 3. Hypothesis Setup (統計假設建立)
* **Null Hypothesis ($H_0$):** $p_1 - p_2 = 0$
  The proportion of current cigarette use is the same between students who felt sad or hopeless and those who did not.
  (感到悲傷或絕望與未感到悲傷或絕望的學生，其吸菸比例在母體中沒有差異。)
* **Alternative Hypothesis ($H_A$):** $p_1 - p_2 \neq 0$
  The proportion of current cigarette use is different between students who felt sad or hopeless and those who did not.
  (兩組學生在母體中的目前吸菸比例存在顯著差異。)
* **Significance Level (顯著水準):** $\alpha = 0.05$

## Part 2: Environmental Setup and Data Loading
## 第二部分：環境建置與資料載入

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

# Set visualization styles / 設定繪圖風格
sns.set_theme(style="whitegrid", context="talk")

# Create Cookiecutter output folders if they do not exist / 建立專案輸出資料夾
os.makedirs("../outputs/figures", exist_ok=True)
os.makedirs("../outputs/tables", exist_ok=True)

# Try loading the cleaned data / 嘗試載入清洗後的資料集
try:
    df = pd.read_csv("../data/processed/cleaned_data.csv")
except FileNotFoundError:
    # Fallback for Google Colab environment / Colab 暫存區備用路徑
    df = pd.read_csv("cleaned_data.csv")

print(f"Data loaded successfully! Effective sample size (n): {len(df)}")

## Part 3: Descriptive Statistics & Contingency Table
## 第三部分：描述性統計與交叉表分析

In [ ]:
# Create the contingency table / 建立交叉表進行計數
contingency_table = pd.crosstab(
    df['SadOrHopeless_Recoded'], 
    df['CurrentCigaretteUse_Recoded'], 
    margins=True, 
    margins_name="Total"
)

print("=== Contingency Table (Observed Counts) ===")
display(contingency_table)

# Extract core parameters for inference / 提取核心統計參數
n1 = contingency_table.loc[1, 'Total']  # Sad Group size
x1 = contingency_table.loc[1, 1]      # Sad & Smoked
p1_hat = x1 / n1                      # Sad Group proportion

n2 = contingency_table.loc[0, 'Total']  # Not Sad Group size
x2 = contingency_table.loc[0, 1]      # Not Sad & Smoked
p2_hat = x2 / n2                      # Not Sad Group proportion

diff_hat = p1_hat - p2_hat            # Sample proportion difference

print("\n=== Descriptive Group Summaries ===")
print(f"Group 1 (Sad): n1 = {n1:,}, Smokers x1 = {x1:,}, Proportion p1_hat = {p1_hat:.4f} ({p1_hat*100:.2f}%)")
print(f"Group 2 (Not Sad): n2 = {n2:,}, Smokers x2 = {x2:,}, Proportion p2_hat = {p2_hat:.4f} ({p2_hat*100:.2f}%)")
print(f"Observed Difference (p1_hat - p2_hat): {diff_hat:.4f} ({diff_hat*100:.2f}%)")

## Part 4: Statistical Assumption Verification
## 第四部分：雙樣本推論統計假設檢查
---
To perform a Two-Proportion Z-test, we must verify the following assumptions:
1. **Independence (獨立性):** The sample is drawn from the YRBS 2007, a complex multi-stage cluster randomized survey. Observations are independent within and between groups. The total sample size ($13,174$) is less than 10% of all US adolescents.
2. **Success/Failure Condition (成功/失敗條件):** Both groups must have at least 10 observed successes and 10 observed failures. This ensures the sampling distribution of $(\hat{p}_1 - \hat{p}_2)$ is approximately normal.

我們必須利用下方的程式碼，動態驗證兩組的「成功與失敗個數」是否皆大於等於 10。

In [ ]:
# Verify Success/Failure Condition / 驗證成功與失敗次數是否符合常態逼近條件
cell_00 = contingency_table.loc[0, 0]
cell_01 = contingency_table.loc[0, 1]
cell_10 = contingency_table.loc[1, 0]
cell_11 = contingency_table.loc[1, 1]

conditions_met = all(cell >= 10 for cell in [cell_00, cell_01, cell_10, cell_11])

print("=== Success/Failure Condition Check ===")
print(f"- Group 0 Failures (No Sad, No Smoke): {cell_00:,} (>=10)")
print(f"- Group 0 Successes (No Sad, Smoke): {cell_01:,} (>=10)")
print(f"- Group 1 Failures (Sad, No Smoke): {cell_10:,} (>=10)")
print(f"- Group 1 Successes (Sad, Smoke): {cell_11:,} (>=10)")
print(f"Is Success/Failure condition met? {'YES! (Valid for Z-test)' if conditions_met else 'NO!'}")

## Part 5: Two-Proportion Z-Test and Confidence Interval
## 第五部分：雙樣本比例 Z 檢定與信賴區間計算

In [ ]:
# 1. Calculate the Pooled Proportion / 計算合併比例
p_pooled = (x1 + x2) / (n1 + n2)
print(f"Pooled Proportion (p_pooled): {p_pooled:.4f} ({p_pooled*100:.2f}%)")

# 2. Calculate the Standard Error for the Hypothesis Test / 計算檢定的標準誤
se_test = np.sqrt(p_pooled * (1 - p_pooled) * (1/n1 + 1/n2))

# 3. Compute Z-statistic and P-value / 計算 Z 統計量與 P 值
z_stat = (p1_hat - p2_hat) / se_test
p_value = stats.norm.sf(abs(z_stat)) * 2  # Two-tailed test

# 4. Calculate the 95% Confidence Interval for the Difference / 計算兩組比例差異的 95% 信賴區間
# Note: CI uses unpooled standard error (使用未合併標準誤計算信賴區間)
se_interval = np.sqrt((p1_hat * (1 - p1_hat) / n1) + (p2_hat * (1 - p2_hat) / n2))
margin_of_error = stats.norm.ppf(0.975) * se_interval
ci_lower = diff_hat - margin_of_error
ci_upper = diff_hat + margin_of_error

print("\n=== Inferential Test Results ===")
print(f"Z-statistic: {z_stat:.4f}")
print(f"P-value: {p_value:.4e} (Scientific Notation) / {p_value:.10f} (Decimal)")
print(f"95% Confidence Interval for the Difference: [{ci_lower*100:.2f}%, {ci_upper*100:.2f}%]")

# 5. Export Summary Table to CSV / 將所有指標整理成 DataFrame 並匯出成 CSV 檔
summary_data = {
    "Metric": [
        "Group 1 (Sad) Sample Size (n1)",
        "Group 1 (Sad) Smokers (x1)",
        "Group 1 (Sad) Proportion (p1_hat)",
        "Group 2 (Non-Sad) Sample Size (n2)",
        "Group 2 (Non-Sad) Smokers (x2)",
        "Group 2 (Non-Sad) Proportion (p2_hat)",
        "Difference in Proportions (p1_hat - p2_hat)",
        "Z-test Statistic",
        "P-value",
        "95% CI Lower Bound",
        "95% CI Upper Bound"
    ],
    "Value": [
        float(n1), float(x1), round(p1_hat, 4),
        float(n2), float(x2), round(p2_hat, 4),
        round(diff_hat, 4), round(z_stat, 4),
        p_value, round(ci_lower, 4), round(ci_upper, 4)
    ]
}
summary_df = pd.DataFrame(summary_data)
summary_df.to_csv("../outputs/tables/summary_table.csv", index=False)
print("\nSummary table exported to 'outputs/tables/summary_table.csv'!")

## Part 6: Data Visualization
## 第六部分：數據與推論結果視覺化

In [ ]:
# ==========================================
# Visualization 1: Proportion Bar Chart with 95% CI
# 圖表一：帶有 95% 信賴區間誤差線的比例比較圖
# ==========================================
plt.figure(figsize=(10, 6))

groups = ['Group 2: Not Sad', 'Group 1: Sad or Hopeless']
proportions = [p2_hat, p1_hat]

# Calculate individual CI standard errors for error bars
# 計算兩組各自的信賴區間半寬度，用作誤差線（Error bars）
err_group2 = stats.norm.ppf(0.975) * np.sqrt(p2_hat * (1 - p2_hat) / n2)
err_group1 = stats.norm.ppf(0.975) * np.sqrt(p1_hat * (1 - p1_hat) / n1)
yerrs = [err_group2, err_group1]

bars = plt.bar(groups, proportions, yerr=yerrs, capsize=10, color=['#3498db', '#e74c3c'], edgecolor='black', width=0.5)

# Add labels on top of the bars / 在長條圖頂端標記百分比
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 0.015, f"{height*100:.2f}%", 
             ha='center', va='bottom', fontweight='bold', fontsize=12)

plt.title('Comparison of Current Smoking Rates with 95% Confidence Intervals', fontweight='bold', fontsize=14)
plt.ylabel('Current Smoking Proportion (p̂)', fontsize=12)
plt.ylim(0, 0.35)
plt.tight_layout()
plt.savefig('../outputs/figures/smoking_rates_comparison.png', dpi=300)
plt.show()

# ==========================================
# Visualization 2: Sampling Distributions & Z-test Visualization
# 圖表二：雙常態抽樣分配與顯著性鴻溝視覺化
# ==========================================
plt.figure(figsize=(12, 6))

# Generate a range for the x-axis / 設定 x 軸區間
x_axis = np.linspace(0.10, 0.35, 1000)

# Standard errors for the sampling distribution of each group
# 計算兩組獨立抽樣分佈的標準誤
se_g2 = np.sqrt(p2_hat * (1 - p2_hat) / n2)
se_g1 = np.sqrt(p1_hat * (1 - p1_hat) / n1)

# Plot normal curves / 繪製兩個常態分佈鐘型曲線
y_g2 = stats.norm.pdf(x_axis, p2_hat, se_g2)
y_g1 = stats.norm.pdf(x_axis, p1_hat, se_g1)

plt.plot(x_axis, y_g2, color='#3498db', linewidth=2.5, label=f'Group 2: Not Sad (Mean={p2_hat*100:.2f}%)')
plt.fill_between(x_axis, y_g2, color='#3498db', alpha=0.15)

plt.plot(x_axis, y_g1, color='#e74c3c', linewidth=2.5, label=f'Group 1: Sad (Mean={p1_hat*100:.2f}%)')
plt.fill_between(x_axis, y_g1, color='#e74c3c', alpha=0.15)

# Add annotations to show the absolute difference / 標記兩組間的實質差距箭頭
plt.annotate('', xy=(p2_hat, 15), xytext=(p1_hat, 15),
             arrowprops=dict(arrowstyle="<->", color='#2c3e50', lw=2))
plt.text((p1_hat + p2_hat)/2, 17, f"Absolute Gap: +{diff_hat*100:.2f}%", 
         ha='center', va='bottom', fontweight='bold', color='#2c3e50', fontsize=12)

plt.title('Sampling Distributions: Visualizing the Group Difference', fontweight='bold', fontsize=14)
plt.xlabel('Estimated Current Smoking Proportion', fontsize=12)
plt.ylabel('Probability Density', fontsize=12)
plt.legend(loc='upper right')
plt.tight_layout()
plt.savefig('../outputs/figures/sampling_distributions_gap.png', dpi=300)
plt.show()

## Part 7: Final Interpretation & Conclusion
## 第七部分：情境解釋與最終結論
---
### 1. Results Summary (成果數值整理)
* **Group 1 (Felt Sad/Hopeless):** Smoking Rate $\hat{p}_1 = 27.61\%$ ($1,064 / 3,854$)
* **Group 2 (Did NOT Feel Sad):** Smoking Rate $\hat{p}_2 = 16.18\%$ ($1,508 / 9,320$)
* **Observed Sample Difference:** $\hat{p}_1 - \hat{p}_2 = 11.43\%$
* **Hypothesis Test:** $Z = 15.0536$, $P = 3.27 \times 10^{-51} < 0.05$ (Highly statistically significant!)
* **95% Confidence Interval for Difference:** $[9.83\%, 13.02\%]$

### 2. Contextual Conclusion (實際情境結論)
Since our P-value ($3.27 \times 10^{-51}$) is practically $0$ and far below $\alpha = 0.05$, we **strongly reject the null hypothesis ($H_0$)**. There is overwhelming empirical evidence to conclude that the proportion of current cigarette use is significantly higher among American adolescents who feel sad or hopeless compared to those who do not.

We are 95% confident that the true population proportion of smoking among adolescents who feel sad or hopeless is between **9.83% and 13.02% higher** than those who do not experience such psychological distress.

由於我們的 P 值遠遠小於 0.05，我們**強烈拒絕虛無假設（Reject $H_0$）**。這證明了「心理健康狀況（感到悲傷或絕望）」與「吸菸行為」之間存在極端顯著的關聯。

### 3. Critical Causation Guardrail (警告：因果關係把關)
**WARNING:** Because the Youth Risk Behavior Survey (YRBS) is an observational, cross-sectional study, we **cannot claim causality** (i.e., we cannot say that feeling sad or hopeless *causes* students to smoke, or vice versa). There may be lurking confounding variables (e.g., family environment, socioeconomic status, peer pressure) that influence both variables. We can only conclude a highly robust **statistical association**.

**警告：** 由於本資料來源為觀察性橫斷面調查（Observational Study），此處結果僅能證實兩者存在強烈的**統計相關性（Association）**，**絕對不代表因果關係（Causality）**。